In [221]:
import pandas as pd
file_path ='MRNGO_features.xlsx'

df1 = pd.read_excel(file_path, sheet_name='features')
df2 = pd.read_excel(file_path, sheet_name='concept_features')

merged_df = pd.merge(df1, df2, on='Feature', how='left')

file_path = 'MRNGO_features_clustered_stimulimaking_preprocessed.xlsx'
df3 = pd.read_excel(file_path)

df_final = pd.merge(merged_df, df3, left_on='Feature', right_on='vector_lemma_C', how='inner')

cols = list(df_final.columns)
cols.insert(cols.index('Feature') + 1, cols.pop(cols.index('vector_lemma_C')))
df_final = df_final[cols]

cols_to_drop = set()
for i in range(len(df_final.columns)):
    col_name_1 = df_final.columns[i]
    if col_name_1 in cols_to_drop:
        continue
    for j in range(i + 1, len (df_final.columns)):
        col_name_2 = df_final.columns[j]
        if df_final.iloc[:, i].equals(df_final.iloc[:, j]):
            cols_to_drop.add(col_name_2)
df_final = df_final.drop(columns=cols_to_drop)

df_final.to_excel('pilot_features.xlsx', sheet_name='sorted_by_features', index=False)


In [1]:
import pandas as pd
import numpy as np
import os 

file_path = 'pilot_features.xlsx'
sheet_name = 'sorted_by_features'
column_to_check = 'itemno'
new_column_name = 'filter_status'

# features that we did not visualized at the end 
items_to_exclude = [ 
    'f00014',
    'f00213',
    'f00217',
    'f00220',
    'f00486',
    'f00497',
    'f00501',
    'f00649',
    'f00991',
    'f01105',
    'f01406',
    'f01528',
    'f01932',
    'f02259'
]

if not os.path.exists(file_path):
    print(f"file was not find")
else:
    try:
        df = pd.read_excel(file_path, sheet_name=sheet_name)
        df[new_column_name] = np.where(
            df[column_to_check].isin(items_to_exclude),
            'excluded',
            'included'
        )

        with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            df.to_excel(writer,sheet_name=sheet_name, index=False)
        print(f"success, sheet has been updated")

    except PermissionError:
        print(f"permission error")

success, sheet has been updated


In [22]:
import pandas as pd
file_path = 'pilot_features.xlsx'
sheet_name = 'sorted_by_features'
sorted_feature_df_main = df[(df['filter_status'] == 'included') & (df['Feature'] != 'HOL_BOLT')]

# filtering out features that were excluded from the set and also the feature 'HOL_BOLT'


In [23]:
import numpy as np

def proportional_10(col, which='top'):
    asc = (which == 'bottom')

    def pick(g):
        vals = sorted(g[col].dropna().unique(), reverse=not asc)
        vals = vals[:10]                              # if >10 distinct values, keep most extreme 10
        sub = g[g[col].isin(vals)]
        cnt = sub[col].value_counts().reindex(vals, fill_value=0)

        if len(vals) == 10:
            n = pd.Series(1, index=vals)             # 1 from each
        else:
            q = cnt / cnt.sum() * 10
            n = np.floor(q).astype(int)
            n[n == 0] = 1                            # include each value
            while n.sum() > 10: n[n.idxmax()] -= 1
            while n.sum() < 10: n[(q - n).idxmax()] += 1

        return pd.concat([sub[sub[col].eq(v)].sample(n=min(n[v], len(sub[sub[col].eq(v)])), random_state=42) for v in vals])

    return sorted_feature_df_main.groupby('Concept', group_keys=False).apply(pick).reset_index(drop=True)



In [10]:
df_freq_x_top10 = df.groupby('Concept').apply(
    lambda x: x.nlargest(10, 'frequency_x')
).reset_index(drop=True)
list_sequence_1 = df_freq_x_top10.groupby('Conceptno')['Feature'].apply(list).reset_index()

print(list_sequence_1)

   Conceptno                                            Feature
0        a05  [RÉSZE_LÁB, ILYEN_NAGY, ILYEN_BARNA, FAJTA_ÁLL...
1        a16  [RÉSZE_LÁB, ILYEN_NAGY, ILYEN_BARNA, HOL_HÁZ, ...
2        a19  [RÉSZE_LÁB, ILYEN_NAGY, ILYEN_BARNA, FAJTA_ÁLL...
3        b02  [ILYEN_BARNA, ILYEN_PIROS, LEHET_OLVAS, ALFAJT...
4        b10  [LEHET_ESZIK, ILYEN_NAGY, RÉSZE_UJJ, ILYEN_HOS...
5        b11  [ILYEN_BARNA, RÉSZE_UJJ, ILYEN_SÁRGA, LEHET_ME...
6        c10  [ILYEN_NAGY, RÉSZE_SZÁR, RÉSZE_BELSŐ, ILYEN_HO...
7        c13  [ILYEN_NAGY, HOL_HÁZ, LEHET_MEGY, RÉSZE_BELSŐ,...
8        c28  [ILYEN_NAGY, RÉSZE_UJJ, RÉSZE_BELSŐ, ILYEN_KIC...
9        f12  [LEHET_ESZIK, FAJTA_GYÜMÖLCS, ILYEN_PIROS, RÉS...
10       f15  [LEHET_ESZIK, FAJTA_GYÜMÖLCS, ILYEN_SÁRGA, ALF...
11       f17  [LEHET_ESZIK, ILYEN_NAGY, HOL_HÁZ, FAJTA_GYÜMÖ...
12       l02  [ILYEN_NAGY, HOL_HÁZ, LEHET_MEGY, HOL_FÖLD, LE...
13       l12  [ILYEN_NAGY, HOL_HÁZ, LEHET_JÁTSZIK, MIBŐL_VAS...
14       l23  [ILYEN_NAGY, ILYEN_BARNA, 

In [11]:
df_freq_x_low10 = df.groupby('Concept').apply(
    lambda x: x.nsmallest(10, 'frequency_x')
).reset_index(drop=True)
list_sequence_2 = df_freq_x_low10.groupby('Conceptno')['Feature'].apply(list).reset_index()

print(list_sequence_2)

   Conceptno                                            Feature
0        a05  [HOL_ÁLLATKERT, ILYEN_ALACSONY, HOL_MESE, ILYE...
1        a16  [ILYEN_BOLDOG, ILYEN_HANGOS, RÉSZE_HAS, TUD_SÉ...
2        a19  [HOL_ÁLLATKERT, ILYEN_BOLDOG, RÉSZE_HAS, TUD_S...
3        b02  [LEHET_NÉZ, RÉSZE_KÖR, TUD_SEGÍT, HOL_MESE, IL...
4        b10  [LEHET_MOZOG, RÉSZE_HÚS, RÉSZE_BŐR, RÉSZE_KÖR,...
5        b11  [HOL_TÉL, LEHET_MOZOG, RÉSZE_BOKA, EMBER_KI, L...
6        c10  [ILYEN_NÉGYZETALAKÚ, NEMILYEN_KOSZOS, RÉSZE_BO...
7        c13  [HOL_KINT, NEMILYEN_KOSZOS, AKCIÓ_VÍZ, MIBŐL_F...
8        c28  [HOL_TÉL, LEHET_MOZOG, RÉSZE_HAS, ILYEN_SZÍNES...
9        f12  [HOL_HŰTŐ, ILYEN_SAVANYKÁS, MÓD_MEGMOS_FOGYASZ...
10       f15  [HOL_HŰTŐ, ILYEN_SAVANYKÁS, BENNE_MAG, HOL_BOK...
11       f17  [ILYEN_SAVANYKÁS, MÓD_MEGMOS_FOGYASZT, BENNE_M...
12       l02  [HOL_KINT, HOL_ÁLLATKERT, ILYEN_PICI, TUD_ÜL, ...
13       l12  [SOK_ABLAK, LEHET_NÉZ, FAJTA_HELY, HOL_SZOBA, ...
14       l23  [BENNE_FÖLD, ILYEN_ALACSON

In [24]:
df_freq_y_top10 = proportional_10('frequency_y', 'top') 
list_sequence_3 = df_freq_y_top10.groupby('Conceptno')['Feature'].apply(list).reset_index()

print(list_sequence_3)

   Conceptno                                            Feature
0        a05  [FAJTA_ÁLLAT, ILYEN_SÁRGA, RÉSZE_FEJ, RÉSZE_NY...
1        a16  [RÉSZE_FAROK, RÉSZE_LÁB, FAJTA_ÁLLAT, NÉGY_LÁB...
2        a19  [FAJTA_ÁLLAT, LEHET_LOVAGOL, RÉSZE_LÁB, HOL_FA...
3        b02  [ILYEN_KÉK, ALFAJTA_ZÖLD, EMBER_RÉSZE, KI_EMBE...
4        b10  [RÉSZE_UJJ, RÉSZE_KÖRÖM, EMBER_RÉSZE, ILYEN_FO...
5        b11  [RÉSZE_LÁBFEJ, LEHET_JÁR, RÉSZE_COMB, FAJTA_TE...
6        c10  [RÉSZE_SZÁR, LEHET_FELVESZ, ALFAJTA_KICSI, KI_...
7        c13  [RÉSZE_TALP, LEHET_FELVESZ, LEHET_HORD, RÉSZE_...
8        c28  [RÉSZE_UJJ, RÉSZE_MINTA, MIBŐL_CÉRNA, ALFAJTA_...
9        f12  [ILYEN_PIROS, FAJTA_GYÜMÖLCS, HOL_FA, TUD_TERE...
10       f15  [ILYEN_SAVANYÚ, ILYEN_SÁRGA, ILYEN_CITROMSÁRGA...
11       f17  [RÉSZE_MAG, ILYEN_PIROS, LEHET_ESZIK, ILYEN_FI...
12       l02  [BENNE_JÁTSZIK, BENNE_JÁTÉK, BENNE_BARÁT, BENN...
13       l12  [HOL_HÁZ, LEHET_JÁTSZIK, BENNE_EMBER, HOL_LAKÁ...
14       l23  [FAJTA_HELY, BENNE_EMBER, 

In [25]:
df_freq_y_low10 = proportional_10('frequency_y', 'bottom')  
list_sequence_4 = df_freq_y_low10.groupby('Conceptno')['Feature'].apply(list).reset_index()

print(list_sequence_4)

   Conceptno                                            Feature
0        a05  [TUD_MEGY, RÉSZE_UJJ, HOL_ÁLLATKERT, ILYEN_ALA...
1        a16  [ILYEN_FINOM, ILYEN_GYORS, HOL_UTCA, ILYEN_NAG...
2        a19  [LEHET_RÁTESZ, RÉSZE_ORR, RÉSZE_SZEM, RÉSZE_FE...
3        b02  [LEHET_JÁTSZIK, HOL_MESE, LEHET_OLVAS, ILYEN_Z...
4        b10  [MIBŐL_BŐR, ILYEN_NAGY, RÉSZE_BŐR, RÉSZE_HÚS, ...
5        b11  [HOL_TÉL, LEHET_HÚZ, RÉSZE_UJJ, LEHET_FELVESZ,...
6        c10  [RÉSZE_BELSŐ, HOL_UTCA, RÉSZE_COMB, NEMILYEN_K...
7        c13  [ILYEN_NAGY, MIVEL_VARR, FAJTA_RUHADARAB, KI_E...
8        c28  [HOL_OTTHON, ILYEN_NAGY, LEHET_MOZOG, ALFAJTA_...
9        f12  [HOL_FÖLD, HOL_PIAC, RÉSZE_MAG, RÉSZE_LEVÉL, I...
10       f15  [RÉSZE_OLDAL, HOL_BOKOR, ALFAJTA_ZÖLD, LEHET_S...
11       f17  [HOL_HÁZ, ILYEN_SAVANYKÁS, ILYEN_EHETŐ, HOL_ER...
12       l02  [ILYEN_PICI, LEHET_MEGY, KI_GYEREK, HOL_HÁZ, H...
13       l12  [BENNE_JÁTÉK, SOK_ABLAK, KI_GYEREK, BENNE_BARÁ...
14       l23  [BENNE_ÍRÁS, ILYEN_ZÖLD, I

In [26]:
df_dist_top10 = proportional_10('Distinct', 'top')
list_sequence_5 = df_dist_top10.groupby('Conceptno')['Feature'].apply(list).reset_index()

print(list_sequence_5)

   Conceptno                                            Feature
0        a05  [ILYEN_CUKI, TUD_MEGY, FAJTA_ÁLLAT, RÉSZE_UJJ,...
1        a16  [RÉSZE_SZÁJ, RÉSZE_ORR, ILYEN_FINOM, ILYEN_PUH...
2        a19  [ILYEN_BOLDOG, RÉSZE_HANG, RÉSZE_ORR, LEHET_JÁ...
3        b02  [TUD_SEGÍT, ILYEN_FONTOS, BENNE_EMBER, HOL_MES...
4        b10  [LEHET_FOG, RÉSZE_KÖR, HOL_TEST, LEHET_ÍR, LEH...
5        b11  [TUD_SEGÍT, EMBER_RÉSZE, RÉSZE_TALP, LEHET_JÁR...
6        c10  [RÉSZE_COMB, LEHET_HORD, HOL_TEST, ILYEN_KÉK, ...
7        c13  [RÉSZE_LÁBFEJ, RÉSZE_TALP, RÉSZE_SAROK, ILYEN_...
8        c28  [LEHET_MOZOG, ILYEN_SZÍNES , RÉSZE_KÉZ, MIBŐL_...
9        f12  [ILYEN_SAVANYÚ, FAJTA_GYÜMÖLCS, ILYEN_FINOM, F...
10       f15  [HOL_HŰTŐ, RÉSZE_MAG, FAJTA_GYÜMÖLCS, TUD_TERE...
11       f17  [ILYEN_SAVANYKÁS, LEHET_RÁTESZ, RÉSZE_SZÁR, IL...
12       l02  [BENNE_JÁTSZIK, BENNE_BARÁT, TUD_ÜL, BENNE_EMB...
13       l12  [BENNE_JÁTSZIK, BENNE_ESZIK, LEHET_NÉZ, HOL_SZ...
14       l23  [LEHET_LOVAGOL, BENNE_BARÁ

In [27]:
df_dist_low10 = proportional_10('Distinct', 'bottom')   # or 'bottom'
list_sequence_6 = df_dist_low10.groupby('Conceptno')['Feature'].apply(list).reset_index()

print(list_sequence_6)

   Conceptno                                            Feature
0        a05  [ILYEN_NAGY, ILYEN_BARNA, ILYEN_KICSI, ILYEN_F...
1        a16  [ILYEN_NAGY, HOL_HÁZ, ILYEN_BARNA, LEHET_JÁTSZ...
2        a19  [ILYEN_NAGY, HOL_PIAC, ILYEN_KICSI, ILYEN_FEHÉ...
3        b02  [RÉSZE_BELSŐ, KI_EMBER, ILYEN_FEHÉR, ILYEN_PIR...
4        b10  [ILYEN_NAGY, ILYEN_HOSSZÚ, KI_EMBER, LEHET_ESZ...
5        b11  [ILYEN_HOSSZÚ, KI_EMBER, LEHET_MEGY, LEHET_HAS...
6        c10  [ILYEN_NAGY, RÉSZE_BELSŐ, LEHET_VESZ, HOL_UTCA...
7        c13  [ILYEN_NAGY, HOL_HÁZ, ILYEN_HOSSZÚ, LEHET_VESZ...
8        c28  [ILYEN_NAGY, RÉSZE_BELSŐ, ILYEN_KICSI, RÉSZE_V...
9        f12  [HOL_PIAC, HOL_FÖLD, HOL_KERT, ILYEN_PIROS, HO...
10       f15  [HOL_PIAC, HOL_FÖLD, ALFAJTA_ZÖLD, LEHET_ESZIK...
11       f17  [ILYEN_NAGY, HOL_HÁZ, RÉSZE_BELSŐ, LEHET_VESZ,...
12       l02  [ILYEN_NAGY, HOL_HÁZ, LEHET_JÁTSZIK, HOL_UTCA,...
13       l12  [ILYEN_NAGY, HOL_HÁZ, LEHET_JÁTSZIK, KI_GYEREK...
14       l23  [ILYEN_NAGY, ALFAJTA_KICSI

In [28]:
df_random = sorted_feature_df_main.groupby('Concept').apply(
    lambda x: x.sample(n=min(len(x), 10))
).reset_index(drop=True)
list_sequence_7 = df_random.groupby('Conceptno')['Feature'].apply(list).reset_index()

print(list_sequence_7)

   Conceptno                                            Feature
0        a05  [RÉSZE_LÁB, RÉSZE_TEST, RÉSZE_FAROK, TUD_MEGY,...
1        a16  [ILYEN_HOSSZÚ, ILYEN_CUKI, RÉSZE_FÜL, RÉSZE_FE...
2        a19  [RÉSZE_FÜL, RÉSZE_KÖRÖM, RÉSZE_FEJ, RÉSZE_ORR,...
3        b02  [ILYEN_ZÖLD, LEHET_OLVAS, ILYEN_KÉK, EMBER_RÉS...
4        b10  [HOL_TEST, LEHET_MOZOG, LEHET_FOG, TUD_FUT, IL...
5        b11  [KI_EMBER, TUD_JÁR, ILYEN_SÁRGA, ILYEN_HOSSZÚ,...
6        c10  [MIBŐL_CÉRNA, ILYEN_NÉGYZETALAKÚ, ILYEN_HOSSZÚ...
7        c13  [LEHET_JÁR, MIVEL_VARR, HOL_OTTHON, MIBŐL_SZÖV...
8        c28  [MIBŐL_ANYAG, MIBŐL_FONAL, LEHET_HORD, HOL_OTT...
9        f12  [HOL_FÖLD, RÉSZE_SZÁR, RÉSZE_MAG, HOL_FA, FAJT...
10       f15  [ILYEN_SAVANYÚ, TUD_TEREM, FAJTA_GYÜMÖLCS, RÉS...
11       f17  [LEHET_RÁTESZ, RÉSZE_KÜLSŐ, RÉSZE_BELSŐ, RÉSZE...
12       l02  [LEHET_MEGY, BENNE_BARÁT, ILYEN_NAGY, HOL_HÁZ,...
13       l12  [ILYEN_NAGY, HOL_LAKÁS, LEHET_NÉZ, BENNE_BARÁT...
14       l23  [LEHET_VEZET, FAJTA_HELY, 

In [30]:
from pathlib import Path
import pandas as pd

out_root = Path("sequences")
feat_map = (sorted_feature_df_main[["Feature", "itemno"]]
            .drop_duplicates("Feature")
            .set_index("Feature")["itemno"])

versions = ["A", "B", "C", "D", "E"]
data_frames = [list_sequence_3, list_sequence_4, list_sequence_5, list_sequence_6, list_sequence_7]

for version, df in zip(versions, data_frames):
    for rowno, row in df.iterrows():
        concept = row["Conceptno"]
        features = row["Feature"]

        concept_dir = out_root / concept
        concept_dir.mkdir(parents=True, exist_ok=True)

        out_df = pd.DataFrame({
            "trialno": [f"{i:02d}" for i in range(1, len(features) + 1)],
            "feature_name": features,
        })
        out_df["feature_file"] = out_df["feature_name"].map(feat_map).map(lambda x: f"stimuli/{x}.png")

        out_df.to_csv(concept_dir / f"{concept}_list{version}.csv", index=False, encoding="utf-8-sig")

In [33]:
from pathlib import Path
import pandas as pd

rng = 42

base = sorted_feature_df_main[["Conceptno", "Concept"]].drop_duplicates("Conceptno").copy()
base["initial"] = base["Conceptno"].str[0]

if base["initial"].nunique() > 10:
    raise ValueError("More than 10 unique first characters, so all cannot be included in a 10-item demo list.")

must_have = base.groupby("initial", group_keys=False).apply(lambda g: g.sample(1, random_state=rng))
rest = base.drop(must_have.index).sample(n=10 - len(must_have), random_state=rng)

demo = pd.concat([must_have, rest]).sample(frac=1, random_state=rng).reset_index(drop=True)

versions = pd.Series(["A","A","B","B","C","C","D","D","E","E"]).sample(frac=1, random_state=rng).reset_index(drop=True)

demo.insert(0, "concept_trial", [f"{i:02d}" for i in range(1, 11)])
demo.insert(1, "version", versions)
demo = demo[["concept_trial", "version", "Conceptno", "Concept"]]
demo.columns = ["concept_trial", "version", "conceptno", "concept"]

Path("concept_lists").mkdir(exist_ok=True)
demo.to_csv("concept_lists/demo.csv", index=False, encoding="utf-8-sig")